In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import holidays
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict
from clase_reportes_new import ReportClassNew
import json
from collections import defaultdict
from send_mail import MailSender
import pandas as pd
ms = MailSender()
rc = ReportClassNew()

In [2]:
# Precio por año de los productos shopi
# shopi = rc.consolidar_carpeta(ruta_carpeta=r'D:\Desktop\shopy', extension='csv')
# shopi['Created at']= pd.to_datetime(shopi['Created at']).dt.to_period('M').dt.to_timestamp().dt.date
# shopi.groupby(['Lineitem sku', 'Created at']).agg(

#     {'Lineitem price': 'min',
#      'Lineitem price': 'max'}

# ).reset_index().to_excel(r'D:\Desktop\shopy_agrupado.xlsx', index=False)
# filtro = shopi[shopi['Lineitem sku'].isin(['PCN20','PCN10'])]
# filtro['Created at']=pd.to_datetime(filtro['Created at']).dt.date

# filtro.to_excel(r'D:\Desktop\shopy.xlsx', index=False)

In [3]:
## Procesar archivos de República Dominicana
import os
import pandas as pd
import numpy as np

carpeta_seleccionada = r'g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\REPÚBLICA DOMINICANA'
lista_dataframes = []


for archivo in os.listdir(carpeta_seleccionada):
    if archivo.endswith(('.xls', '.xlsx')):   # acepta ambos tipos
        ruta_completa = os.path.join(carpeta_seleccionada, archivo)
        try:
            print(f"Procesando archivo: {ruta_completa}")
            archivo1 = pd.read_excel(ruta_completa, header=None)
          
            # Encontrar el índice de la fila que contiene "Total" en la primera columna
            index_total = int(archivo1[archivo1.iloc[:, 0:1].iloc[:, 0] == 'Total'].index[0])
            archivo1.columns =     archivo1[index_total-2:index_total-1].values[0]
            index_fechas = [i for i in range(len(archivo1.columns)) if pd.notna(archivo1.columns[i])]
            index_fechas.insert(0, 0)

            archivo1= archivo1[index_total+1:].iloc[:, index_fechas]
            meses = {'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
                    'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12}
            archivo1.rename(columns={archivo1.columns[0]: 'Detalles'}, inplace=True)
            archivo1['Clientes'] = np.where(
                archivo1['Detalles'].str.contains('[', regex=False, na=False),
                np.nan,
                archivo1['Detalles']
            )
            print(f"Archivo '{archivo}' procesado correctamente.")
            archivo1['Clientes'] = archivo1['Clientes'].ffill()
            archivo1 = archivo1[archivo1['Detalles'].str.contains('[', regex=False, na=False)]
            archivo1 = archivo1.melt(id_vars=['Clientes', 'Detalles'], var_name='Fecha', value_name='Valor')
            archivo1['Fecha'] = [
                pd.to_datetime(f'{y}-{meses[mes]}-01')
                for mes, y in (i.split(' ') for i in archivo1['Fecha'])
            ]
            
            archivo1['Valor'] = pd.to_numeric(archivo1['Valor'], errors='coerce')
            archivo1 = archivo1[archivo1['Valor']>0 ]
            archivo1['Sucursal'] = archivo1['Clientes'].str.split(',').str[1]
            archivo1.loc[
                archivo1['Clientes']
                .str.split('-').str[1]
                .str.lower()
                .str.contains(r'pocion|\.do|whats|n.do', na=False),
                'Segmento'
            ] = 'Ecommerce'
            archivo1.loc[
                archivo1['Clientes']
                .str.lower()
                .str.contains(r'pocion|\.do|whats|n.do|what|WHAT', na=False),
                'Segmento'
            ] = 'Ecommerce'
            archivo1.loc[
                archivo1['Clientes']
                .str.lower()
                .str.contains(r'farmacia', na=False),
                'Segmento'
            ] = 'Farmacia'
            archivo1.loc[~archivo1['Clientes'].str.contains(',|-'), 'Segmento'] = 'Mayorista'
            archivo1.loc[archivo1['Clientes'].str.lower().str.contains('byg|presencial'), 'Segmento'] = 'BYG'
            archivo1['Clientes'] = archivo1['Clientes'].str.split(',').str[0]
            archivo1['Detalles'] = archivo1['Detalles'].str.strip().str.replace(r'\s+', ' ', regex=True)
            archivo1['Clientes'] = archivo1['Clientes'].str.strip().str.replace(r'\s+', ' ', regex=True)
            archivo1['Valor'] = archivo1['Valor'].astype(int)
            lista_dataframes.append(archivo1)
        except Exception as e:
            print(f"No se pudo leer el archivo '{archivo}'. Error: {e}")



print("Concatenando todos los archivos...")
if lista_dataframes:

    df_concatenado = pd.concat(lista_dataframes, ignore_index=True, )
    df_concatenado = df_concatenado.loc[:, ~df_concatenado.columns.duplicated()]
    df_concatenado = df_concatenado[df_concatenado['Clientes'].notna()]
    print(f"¡Consolidación completada! Total filas: {len(df_concatenado)}")



Procesando archivo: g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\REPÚBLICA DOMINICANA\SELL IN FEBRERO.xlsx
Archivo 'SELL IN FEBRERO.xlsx' procesado correctamente.
Procesando archivo: g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\REPÚBLICA DOMINICANA\SELL IN ENERO.xlsx
Archivo 'SELL IN ENERO.xlsx' procesado correctamente.
Procesando archivo: g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\REPÚBLICA DOMINICANA\SELL IN 2025.xlsx
Archivo 'SELL IN 2025.xlsx' procesado correctamente.
Procesando archivo: g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\REPÚBLICA DOMINICANA\SELL IN MARZO.xlsx
Archivo 'SELL IN MARZO.xlsx' procesado correctamente.
Procesando archivo: g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\REPÚBLICA DOMINICANA\SELL IN ABRIL.xlsx
Archivo 'SELL IN ABRIL.xlsx' procesado correctamente.
Concatenando todos los archivos...
¡Consolidación completada! Total filas: 3262


In [4]:
## Procesar archivos de República Dominicana
import os
import pandas as pd
import numpy as np

In [5]:
archivo1 = pd.read_excel(r'G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\REPÚBLICA DOMINICANA\SELL IN ABRIL.xlsx', header=None)
        
# Encontrar el índice de la fila que contiene "Total" en la primera columna
index_total = int(archivo1[archivo1.iloc[:, 0:1].iloc[:, 0] == 'Total'].index[0])
archivo1.columns =     archivo1[index_total-2:index_total-1].values[0]
index_fechas = [i for i in range(len(archivo1.columns)) if pd.notna(archivo1.columns[i])]
index_fechas.insert(0, 0)

archivo1= archivo1[index_total+1:].iloc[:, index_fechas]
meses = {'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
        'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12}
archivo1.rename(columns={archivo1.columns[0]: 'Detalles'}, inplace=True)
archivo1['Clientes'] = np.where(
    archivo1['Detalles'].str.contains('[', regex=False, na=False),
    np.nan,
    archivo1['Detalles']
)
archivo1['Clientes'] = archivo1['Clientes'].ffill()
archivo1 = archivo1[archivo1['Detalles'].str.contains('[', regex=False, na=False)]
archivo1 = archivo1.melt(id_vars=['Clientes', 'Detalles'], var_name='Fecha', value_name='Valor')


In [6]:
archivo

'desktop.ini'

In [9]:
archivo1['Fecha']

0     2026-04-01
1     2026-04-01
2     2026-04-01
3     2026-04-01
4     2026-04-01
         ...    
359   2026-04-01
360   2026-04-01
361   2026-04-01
362   2026-04-01
363   2026-04-01
Name: Fecha, Length: 364, dtype: datetime64[ns]

In [8]:
archivo1['Fecha'] = [
    pd.to_datetime(f'{y}-{meses[mes]}-01')
    for mes, y in (i.split(' ') for i in archivo1['Fecha'].str.lower())
]


In [ ]:

archivo1['Valor'] = pd.to_numeric(archivo1['Valor'], errors='coerce')
archivo1 = archivo1[archivo1['Valor']>0 ]
archivo1['Sucursal'] = archivo1['Clientes'].str.split(',').str[1]
archivo1.loc[
    archivo1['Clientes']
    .str.split('-').str[1]
    .str.lower()
    .str.contains(r'pocion|\.do|whats|n.do', na=False),
    'Segmento'
] = 'Ecommerce'
archivo1.loc[
    archivo1['Clientes']
    .str.lower()
    .str.contains(r'pocion|\.do|whats|n.do|what|WHAT', na=False),
    'Segmento'
] = 'Ecommerce'
archivo1.loc[
    archivo1['Clientes']
    .str.lower()
    .str.contains(r'farmacia', na=False),
    'Segmento'
] = 'Farmacia'
archivo1.loc[~archivo1['Clientes'].str.contains(',|-'), 'Segmento'] = 'Mayorista'
archivo1.loc[archivo1['Clientes'].str.lower().str.contains('byg|presencial'), 'Segmento'] = 'BYG'
archivo1['Clientes'] = archivo1['Clientes'].str.split(',').str[0]
archivo1['Detalles'] = archivo1['Detalles'].str.strip().str.replace(r'\s+', ' ', regex=True)
archivo1['Clientes'] = archivo1['Clientes'].str.strip().str.replace(r'\s+', ' ', regex=True)
archivo1['Valor'] = archivo1['Valor'].astype(int)

In [ ]:
# ## Procesar archivos de Perú
# import os
# import pandas as pd

# carpeta_seleccionada = r'g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\PERÚ'
# lista_dataframes = []

# for archivo in os.listdir(carpeta_seleccionada):
#     if archivo.endswith(('.xls', '.xlsx')):   # acepta ambos tipos
#         ruta_completa = os.path.join(carpeta_seleccionada, archivo)
#         try:
#             df_peru = pd.read_excel(ruta_completa)

#             # df_peru = (
#             #     df_peru
#             #     .melt(id_vars='SKU', var_name='Fecha', value_name='Ventas')
#             #     .fillna({'Ventas': 0})
#             # )
#             # df_peru['Fecha'] = pd.to_datetime(df_peru['Fecha'], errors='coerce')
#             lista_dataframes.append(df_peru)
#         except Exception as e:
#             print(f"No se pudo leer el archivo '{archivo}'. Error: {e}")



# print("Concatenando todos los archivos...")
# if lista_dataframes:

#     df_concatenado = pd.concat(lista_dataframes, ignore_index=False, axis=0)
#     df_concatenado = df_concatenado.loc[:, ~df_concatenado.columns.duplicated()]
#     print(f"¡Consolidación completada! Total filas: {len(df_concatenado)}")


Concatenando todos los archivos...
¡Consolidación completada! Total filas: 5691


In [ ]:
# ## Correos de contabilidad
# ## Cambiar ruta de archivo y origen de pdfs
# df = pd.read_excel(r'd:\Downloads\LISTADO 2 WILLIAM.xlsx')
# from pathlib import Path

# # Configuración de rutas según tu estructura
# ruta_consolidado = Path(rc.validar_ruta()) / 'certificados contables' / 'reportes'

# html_formato = """
# <html>
# <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333;">
#     <div style="max-width: 620px; margin: auto; border: 1px solid #e6e6e6; padding: 24px; border-radius: 6px;">
        
#         <h2 style="color: #000; margin-top: 0;">Certificado de Retenciones Año Gravable 2025</h2>
        
#         <p>Cordial saludo,</p>
        
#         <p>
#         Por medio del presente nos permitimos remitir el 
#         <strong>Certificado de Retención en la Fuente correspondiente al año gravable 2025</strong>, 
#         expedido en cumplimiento de lo establecido en el 
#         <strong>artículo 381 del Estatuto Tributario</strong>, el cual establece la obligación 
#         de expedir y entregar los certificados de las retenciones practicadas.
#         </p>
        
#         <p>
#         Agradecemos confirmar la correcta recepción del documento adjunto. 
#         En caso de que este mensaje no corresponda a su área o dependencia, 
#         le agradecemos amablemente remitirlo a la persona o área encargada.
#         </p>
        
#         <p>
#         Quedamos atentos a cualquier inquietud o solicitud adicional.
#         </p>

#         <br>

#         <hr style="border: 0; border-top: 1px solid #eee; margin: 20px 0;">
        
#         <p style="font-size: 12px; color: #777; margin-bottom: 0;">
#         Este es un envío automático generado por <strong>La Poción – Departamento de Contabilidad</strong>.<br>
#         Si encuentra alguna inconsistencia o requiere información adicional, 
#         puede responder directamente a este correo.
#         </p>

#     </div>
# </body>
# </html>
# """

# for index, fila in df.iterrows():
#     nit = str(fila['NIT']).strip()
#     correo_cliente = fila['Correo']
    
#     # Definimos la ruta específica del cliente (la carpeta con su NIT)
#     carpeta_cliente = ruta_consolidado / nit
    
#     # Validación de seguridad: ¿Existe la carpeta para este NIT?
#     if carpeta_cliente.exists() and carpeta_cliente.is_dir():
#         # Obtenemos todos los archivos dentro de la carpeta del cliente
#         # Nueva lógica de captura filtrando archivos basura del sistema
#         adjuntos_cliente = [
#             str(f) for f in carpeta_cliente.iterdir() 
#             if f.is_file() and f.name.lower() != 'desktop.ini' and not f.name.startswith('.')
#         ]
        
#         if not adjuntos_cliente:
#             print(f"⚠️ Advertencia: Carpeta vacía para NIT {nit}. Saltando envío.")
#             continue

#         # Ejecución del envío
#         try:
#             ms.enviar_correo(
#                 destinatarios=correo_cliente,
#                 cc=['vmarquez@lapocion.com'],
#                 asunto=f"Certificados Contables La Poción - NIT {nit}",
#                 cuerpo_html = html_formato,
#                 adjuntos=adjuntos_cliente
                
#             )
#             print(f"✅ Correo enviado con éxito a: {nit}")
#         except Exception as e:
#             print(f"❌ Error al enviar a {nit}: {e}")
#     else:
#         print(f"⚠️ Error: No se encontró carpeta para el NIT {nit} en la ruta {ruta_consolidado}")

📨 Enviando correo a: almoca08@hotmail.es
✅ Correo enviado correctamente.
✅ Correo enviado con éxito a: 1144143406
📨 Enviando correo a: stephaniegomez15@gmail.com
✅ Correo enviado correctamente.
✅ Correo enviado con éxito a: 1144090060
📨 Enviando correo a: camilocespedes12@hotmail.es
✅ Correo enviado correctamente.
✅ Correo enviado con éxito a: 1144075832
📨 Enviando correo a: BRAYANARBOLEDA0201@GMAIL.COM
✅ Correo enviado correctamente.
✅ Correo enviado con éxito a: 1144070107
📨 Enviando correo a: jeca@isabella@gmail.com
✅ Correo enviado correctamente.
✅ Correo enviado con éxito a: 1144060170
📨 Enviando correo a: jenylorrein@gmail.com
✅ Correo enviado correctamente.
✅ Correo enviado con éxito a: 1143926763
📨 Enviando correo a: diegozapata9908@gmail.com
✅ Correo enviado correctamente.
✅ Correo enviado con éxito a: 1143876735
📨 Enviando correo a: JULIANLOZANOH026@GMAIL.COM
✅ Correo enviado correctamente.
✅ Correo enviado con éxito a: 1143853658
📨 Enviando correo a: valentina.cabal@gmail.co

In [ ]:
# import os
# from PyPDF2 import PdfMerger

# def unir_pdfs(ruta_carpeta, nombre_salida):
#     merger = PdfMerger()
    
#     # Listar y filtrar solo archivos PDF, luego ordenar alfabéticamente
#     archivos = [f for f in os.listdir(ruta_carpeta) if f.endswith('.pdf')]
#     archivos.sort() 

#     if not archivos:
#         print("No se encontraron archivos PDF en la carpeta.")
#         return

#     for archivo in archivos:
#         ruta_completa = os.path.join(ruta_carpeta, archivo)
#         print(f"Agregando: {archivo}")
#         merger.append(ruta_completa)

#     # Guardar el resultado final
#     merger.write(nombre_salida)
#     merger.close()
#     print(f"\n--- Éxito: Archivo '{nombre_salida}' generado correctamente ---")

# # Configuración
# carpeta_pdds = r'G:\Unidades compartidas\William\Varios pcn Will porfa unir en un solo pdf'  # Ajusta tu ruta aquí
# archivo_final = 'Reporte_Consolidado_LaPocion.pdf'

# unir_pdfs(carpeta_pdds, archivo_final)

Agregando: Asientos contables (10).pdf
Agregando: Asientos contables (100).pdf
Agregando: Asientos contables (11).pdf
Agregando: Asientos contables (12).pdf
Agregando: Asientos contables (13).pdf
Agregando: Asientos contables (14).pdf
Agregando: Asientos contables (15).pdf
Agregando: Asientos contables (16).pdf
Agregando: Asientos contables (17).pdf
Agregando: Asientos contables (18).pdf
Agregando: Asientos contables (19).pdf
Agregando: Asientos contables (2).pdf
Agregando: Asientos contables (20).pdf
Agregando: Asientos contables (21).pdf
Agregando: Asientos contables (22).pdf
Agregando: Asientos contables (23).pdf
Agregando: Asientos contables (24).pdf
Agregando: Asientos contables (25).pdf
Agregando: Asientos contables (26).pdf
Agregando: Asientos contables (27).pdf
Agregando: Asientos contables (28).pdf
Agregando: Asientos contables (29).pdf
Agregando: Asientos contables (3).pdf
Agregando: Asientos contables (30).pdf
Agregando: Asientos contables (31).pdf
Agregando: Asientos contab

In [ ]:
## Agrupar cuentas contabilidad
# ventas = rc.consolidar_carpeta(ruta_carpeta=r'g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\CLEAN DATA', extension='csv', sep=';', decimal=',')
# ventas['FECHA_FACTURA'] = pd.to_datetime(ventas['FECHA_FACTURA'])
# ventas = ventas[ventas['FECHA_FACTURA'].dt.year >=2025]
# ventas.to_excel(r'd:\Desktop\ventas.xlsx')

# ventas_filtradas= ventas[['CATEGORÍA','CLIENTE']].drop_duplicates()
# ventas_filtradas = ventas_filtradas[~ventas_filtradas['CATEGORÍA'].isin(['SHOPIFY','Proveedores', 'CALL CENTER'])]
# ventas_filtradas.to_excel(r'd:\Desktop\categorias_cliente.xlsx')
# df_mayor = pd.read_excel(r'd:\Downloads\libro_mayor (36).xlsx')
# df_mayor = df_mayor[df_mayor['Unnamed: 3'].notna()]
# # Usar la primera fila como encabezado
# df_mayor.columns = df_mayor.iloc[0]

# # Eliminar la primera fila (ya que ahora es el header)
# df_mayor = df_mayor[1:].reset_index(drop=True)
# df_mayor['Period'] = pd.to_datetime(df_mayor['Fecha']).dt.to_period('M')
# df_mayor_filtrado = df_mayor[df_mayor['Debe  ']>0]
# df_mayor_filtrado = df_mayor_filtrado.groupby(['Period', 'Documento','Contacto'])['Debe  '].sum().reset_index()
# df_mayor_filtrado.to_excel(r'd:\Desktop\nomina_agru.xlsx', index=False)




# Paises

In [9]:
# # Procesa Ecuador
# import os
# import pandas as pd

# carpeta_seleccionada = r'g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\ECUADOR'
# lista_dataframes = []

# for archivo in os.listdir(carpeta_seleccionada):
#     if archivo.endswith(('.xls', '.xlsx')):   # acepta ambos tipos
#         ruta_completa = os.path.join(carpeta_seleccionada, archivo)
#         try:
#             print(f"Leyendo: {archivo} ...")
#             df = pd.read_excel(ruta_completa)
            
#             # Evita error si no existe la columna
#             if 'Fecha' in df.columns:
#                 df = df[df['Fecha'].notna()]
            
#             if {'Tipo', 'Total Uni.', 'Total'}.issubset(df.columns):
#                 df.loc[df['Tipo'] == 'DEV', ['Total Uni.', 'Total']] *= -1
            
#             #  Corrige nombre mal escrito
#             df.rename(columns={'Identificaión': 'Identificación'}, inplace=True)
            
#             if 'Identificación' in df.columns:
#                 df['Identificación'] = (
#                     df['Identificación'].astype(str).replace(r'\D+', '', regex=True)
#                 )
            
#             lista_dataframes.append(df)
#             print(f"Cargado: {archivo} ({len(df)} filas)")
        
#         except Exception as e:
#             print(f"No se pudo leer el archivo '{archivo}'. Error: {e}")

# print("Concatenando todos los archivos...")
# if lista_dataframes:
#     df_concatenado = pd.concat(lista_dataframes, ignore_index=True)
#     print(f"¡Consolidación completada! Total filas: {len(df_concatenado)}")


In [10]:
# ## Procesar archivos de República Dominicana
# import os
# import pandas as pd
# import numpy as np

# carpeta_seleccionada = r'g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\REPÚBLICA DOMINICANA'
# lista_dataframes = []

# for archivo in os.listdir(carpeta_seleccionada):
#     if archivo.endswith(('.xls', '.xlsx')):   # acepta ambos tipos
#         ruta_completa = os.path.join(carpeta_seleccionada, archivo)
#         try:
#             archivo1 = pd.read_excel(ruta_completa, header=None)
#             # Encontrar el índice de la fila que contiene "Total" en la primera columna
#             index_total = int(archivo1[archivo1.iloc[:, 0:1].iloc[:, 0] == 'Total'].index[0])
#             archivo1.columns =     archivo1[index_total-2:index_total-1].values[0]
#             index_fechas = [i for i in range(len(archivo1.columns)) if pd.notna(archivo1.columns[i])]
#             index_fechas.insert(0, 0)

#             archivo1= archivo1[index_total+1:].iloc[:, index_fechas]
#             meses = {'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
#                     'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12}
#             archivo1.rename(columns={archivo1.columns[0]: 'Detalles'}, inplace=True)
#             archivo1['Clientes'] = np.where(
#                 archivo1['Detalles'].str.contains('[', regex=False, na=False),
#                 np.nan,
#                 archivo1['Detalles']
#             )
#             archivo1['Clientes'] = archivo1['Clientes'].ffill()
#             archivo1 = archivo1[archivo1['Detalles'].str.contains('[', regex=False, na=False)]
#             archivo1 = archivo1.melt(id_vars=['Clientes', 'Detalles'], var_name='Fecha', value_name='Valor')
#             archivo1['Fecha'] = [
#                 pd.to_datetime(f'{y}-{meses[mes]}-01')
#                 for mes, y in (i.split(' ') for i in archivo1['Fecha'])
#             ]
#             archivo1['Valor'] = pd.to_numeric(archivo1['Valor'], errors='coerce')
#             archivo1 = archivo1[archivo1['Valor']>0 ]
#             archivo1['Sucursal'] = archivo1['Clientes'].str.split(',').str[1]
#             archivo1.loc[
#                 archivo1['Clientes']
#                 .str.split('-').str[1]
#                 .str.lower()
#                 .str.contains(r'pocion|\.do|whats', na=False),
#                 'Segmento'
#             ] = 'Ecommerce'
#             archivo1.loc[~archivo1['Clientes'].str.contains(',|-'), 'Segmento'] = 'Mayorista'
#             archivo1.loc[archivo1['Clientes'].str.lower().str.contains('byg|presencial'), 'Segmento'] = 'BYG'
#             archivo1['Clientes'] = archivo1['Clientes'].str.split(',').str[0]
#             archivo1['Detalles'] = archivo1['Detalles'].str.strip().str.replace(r'\s+', ' ', regex=True)
#             archivo1['Clientes'] = archivo1['Clientes'].str.strip().str.replace(r'\s+', ' ', regex=True)
#             archivo1['Valor'] = archivo1['Valor'].astype(int)
#             lista_dataframes.append(archivo1)
#         except Exception as e:
#             print(f"No se pudo leer el archivo '{archivo}'. Error: {e}")



# print("Concatenando todos los archivos...")
# if lista_dataframes:

#     df_concatenado = pd.concat(lista_dataframes, ignore_index=True, )
#     df_concatenado = df_concatenado.loc[:, ~df_concatenado.columns.duplicated()]
#     df_concatenado = df_concatenado[df_concatenado['Clientes'].notna()]
#     print(f"¡Consolidación completada! Total filas: {len(df_concatenado)}")


In [11]:
# ## Procesar archivos de Perú
# import os
# import pandas as pd

# carpeta_seleccionada = r'g:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\paises\PERÚ'
# lista_dataframes = []

# for archivo in os.listdir(carpeta_seleccionada):
#     if archivo.endswith(('.xls', '.xlsx')):   # acepta ambos tipos
#         ruta_completa = os.path.join(carpeta_seleccionada, archivo)
#         try:
#             df_peru = pd.read_excel(ruta_completa)

#             # df_peru = (
#             #     df_peru
#             #     .melt(id_vars='SKU', var_name='Fecha', value_name='Ventas')
#             #     .fillna({'Ventas': 0})
#             # )
#             # df_peru['Fecha'] = pd.to_datetime(df_peru['Fecha'], errors='coerce')
#             lista_dataframes.append(df_peru)
#         except Exception as e:
#             print(f"No se pudo leer el archivo '{archivo}'. Error: {e}")



# print("Concatenando todos los archivos...")
# if lista_dataframes:

#     df_concatenado = pd.concat(lista_dataframes, ignore_index=False, axis=1)
#     df_concatenado = df_concatenado.loc[:, ~df_concatenado.columns.duplicated()]
#     print(f"¡Consolidación completada! Total filas: {len(df_concatenado)}")
